### Machine Learning Pipelines

* **Definition:** Pipelines is a mechanism that chains together multiple steps so that the output of one step is used as an input to next step.
* **Benefit:** They make it easy to apply the same preprocessing to train and test.

# **Pre processing and model training**

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [5]:
df = pd.read_csv('titanic.csv')

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### droping every col that isnt usefull to train

In [8]:
df.drop(['PassengerId','Name','Ticket','Cabin'],axis=1, inplace=True)

### Understanding `inplace=True` in Pandas

When dropping columns from a DataFrame, there are two primary ways to make the changes stick:

```python
# Method 1: Using inplace=True
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Method 2: Reassigning to the variable
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

In [9]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [10]:
### Step 1: Train test split

X_train,X_test,Y_train,Y_test = train_test_split(df.drop(columns=['Survived']),
                                                df['Survived'],
                                                test_size=0.2,
                                                random_state=0)

In [11]:
# Applying imputation to get rid of null values

si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.fit_transform(X_test[['Age']])
X_test_embarked = si_embarked.fit_transform(X_test[['Embarked']])

In [12]:
# Apply OHE to to sex,Embarked

ohe_sex = OneHotEncoder(sparse_output=False,drop='first',dtype=int)
ohe_embarked = OneHotEncoder(sparse_output=False,drop='first',dtype=int)

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.fit_transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.fit_transform(X_test_embarked)

In [13]:
X_train_embarked

array([[0, 0],
       [0, 1],
       [0, 0],
       ...,
       [1, 0],
       [0, 1],
       [0, 1]], shape=(712, 2))

## Now since all the indevisual transformations are done lets add each of them together 

In [14]:
### Before concatenating lets first take the already healthy columns which didnt require any transformation

In [15]:
X_train_healthy = X_train.drop(columns=['Age','Embarked','Sex'])
X_test_healthy = X_test.drop(columns=['Age','Embarked','Sex'])

In [16]:
X_train_transformed = np.concatenate((X_train_healthy,X_train_age,X_train_embarked,X_train_sex),axis=1)
X_test_transformed = np.concatenate((X_test_healthy,X_test_age,X_test_embarked,X_test_sex), axis=1)

## **DONE: Final X train and test DFs are ready (X_train_transformed, X_test_transformed)**

In [17]:
X_test_transformed.shape

(179, 8)

## If you have studied 'Column transformer' you must be wondering why arnt we using it here, reason below

In [18]:
from sklearn.compose import ColumnTransformer

In [19]:
ct = ColumnTransformer(transformers=[
    ('t1',SimpleImputer(),['Age']),
    ('t2',SimpleImputer(strategy='most_frequent'),['Embarked']),
    ('t3',OneHotEncoder(),['Embarked']) ### RIGHT HERE: this required the output of t2 as its input
])

In ct u cant use the output of previous transformation as an input to the next transformation in ct

# Model training

In [20]:
model = DecisionTreeClassifier()
model.fit(X_train_transformed,Y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [21]:
pred = model.predict(X_test_transformed) #predict() takes the 'x test' data and gives all the predictions(predicted results)

In [22]:
from sklearn.metrics import accuracy_score
score = accuracy_score(Y_test,pred)   #accuracy_score() takes y test(actual results) and (predicted results) and checks how much predicted results match actual result
print(f'{score*100:.2f} % accurate')

79.33 % accurate


In [23]:
import pickle

In [24]:
pickle.dump(ohe_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(model,open('models/model.pkl','wb'))

### Pickle.dump: How to use

```python
Pickle.dump(<object to save>, open('<address to save into>','wb'))  wb = write binary

What now?: 
Later we can use this model in any application to take inputs from user and output the prediction 

# END ----

In [30]:
t = pd.DataFrame(X_test_transformed)

In [35]:
t.sample(20)

,0,1,2,3,4,5,6,7
39,2.0,0.0,0.0,14.0000,54.000000,0.0,1.0,1.0
83,3.0,0.0,0.0,7.7750,16.000000,0.0,1.0,1.0
142,3.0,0.0,0.0,8.0500,29.515175,0.0,1.0,1.0
87,3.0,0.0,0.0,9.5875,63.000000,0.0,1.0,0.0
1,3.0,0.0,0.0,7.5500,29.515175,0.0,1.0,1.0
127,3.0,1.0,0.0,7.7750,25.000000,0.0,1.0,1.0
11,2.0,0.0,0.0,13.0000,40.000000,0.0,1.0,0.0
140,2.0,0.0,0.0,13.0000,28.000000,0.0,1.0,1.0
57,3.0,4.0,2.0,31.2750,11.000000,0.0,1.0,0.0
14,1.0,0.0,0.0,83.1583,24.000000,0.0,0.0,0.0


In [37]:
t.describe()

,0,1,2,3,4,5,6,7
count,179.000000,179.000000,179.000000,179.000000,179.000000,179.000000,179.000000,179.000000
mean,2.273743,0.497207,0.340782,33.561615,29.515175,0.072626,0.709497,0.625698
std,0.846628,0.938408,0.742802,48.008223,12.682859,0.260249,0.455268,0.485300
min,1.000000,0.000000,0.000000,0.000000,0.420000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,7.972900,22.000000,0.000000,0.000000,0.000000
50%,3.000000,0.000000,0.000000,14.000000,29.515175,0.000000,1.000000,1.000000
75%,3.000000,1.000000,0.000000,30.500000,34.000000,0.000000,1.000000,1.000000
max,3.000000,5.000000,5.000000,263.000000,64.000000,1.000000,1.000000,1.000000


0 = pclass
1 = sibsp
2 = parch
3 = fare
4 = age
5 = embarked
6 = embarked
7 = embarked